# Here we'll make the best , most , complete use of all the concepts 

### 1. Import Everything

In [5]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import r2_score
from sklearn import set_config

import joblib

## 2. Creating Complex Dataset

In [8]:
def load_complex_data():
    data = fetch_california_housing(as_frame=True)
    df = data.frame
    
    # Add categorical feature
    df["IncomeCategory"] = pd.cut(df["MedInc"],
                                  bins=[0,2,4,6,8, np.inf],
                                  labels=["VeryLow","Low","Medium","High","VeryHigh"])
    
    # Add fake text column
    df["Description"] = (
        "House with " +
        df["AveRooms"].astype(str) +
        " rooms located in block " +
        df["AveOccup"].astype(str)
    )
    
    # Introduce missing values
    df.loc[df.sample(frac=0.1).index, "AveRooms"] = np.nan
    
    return df

### 3. Creating a Custom Transformer - Rare Category Grouper

In [11]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    
    def __init__(self, threshold=0.05):
        self.threshold = threshold
    
    def fit(self, X, y=None):
        self.freqs_ = {}
        for col in X.columns:
            freq = X[col].value_counts(normalize=True)
            self.freqs_[col] = freq
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            rare = self.freqs_[col][self.freqs_[col] < self.threshold].index
            X[col] = np.where(X[col].isin(rare), "Rare", X[col])
        return X

### 4. Define Feature Groups

In [14]:
df = load_complex_data()

target = "MedHouseVal"
X = df.drop(columns=[target])
y = df[target]

num_cols = X.select_dtypes(include=["float64"]).columns.tolist()
cat_cols = ["IncomeCategory"]
text_col = "Description"

## 5. Build Sub-Pipelines 

### Numerical Pipeline - Advance Automation Done

In [18]:
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("feature_selection", SelectKBest(score_func=f_regression, k=10))
])

### Categorical Pipeline

In [21]:
cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("rare", RareCategoryGrouper(threshold=0.05)),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

### Text Pipeline - NLP inside ColumnTransformer

In [24]:
text_pipeline = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(max_features=50))
])

## 6. Full Column Transformer

In [28]:
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
    ("text", text_pipeline, text_col)
], remainder="drop")

# 7. Full Pipeline - Model Agnostic

In [31]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge())
], memory="cache_folder", verbose=True)

### 8. Hyperparameter Tuning - Multi-Model

In [35]:
param_grid = [
    {
        "model": [Ridge()],
        "model__alpha": [0.1, 1, 10],
        "preprocessor__num__poly__degree": [1,2],
        "preprocessor__num__feature_selection__k": [5,10]
    },
    {
        "model": [RandomForestRegressor()],
        "model__n_estimators": [100,200],
        "model__max_depth": [None,10]
    },
    {
        "model": [GradientBoostingRegressor()],
        "model__n_estimators": [100],
        "model__learning_rate": [0.01,0.1]
    }
]

### 9. Train 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("Best Model:", grid.best_estimator_)
print("Best Params:", grid.best_params_)

y_pred = grid.predict(X_test)
print("R2 Score:", r2_score(y_test, y_pred))

### 10. Save Full Pipeline

In [ ]:
joblib.dump(grid.best_estimator_, "complex_pipeline_model.pkl")

### 11. Visualize the Pipeline Graph

In [44]:
set_config(display='diagram')
pipeline

Pipeline(memory='cache_folder',
         steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('poly',
                                                                   PolynomialFeatures(include_bias=False)),
                                                                  ('scaler',
                                                                   StandardScaler()),
                                                                  ('feature_selection',
                                                                   SelectKBest(score_func=<function f_regression at 0x000001AEA2859580>))]),
                                                  ['MedInc', 'HouseAge',
                                                   'AveRoom...s',
                                                   'Population', 'AveOccup',
                                                   'Latitude', 'Longitude']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('rare',
                                                                   RareCategoryGrouper()),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['IncomeCategory']),
                                                 ('text',
                                                  Pipeline(steps=[('tfidf',
                                                                   TfidfVectorizer(max_features=50))]),
                                                  'Description')])),
                ('model', Ridge())],
         verbose=True)